# Numerai 13F-HR Data Updater (Ticker Auto-Search Version)

このノートブックは、SECから最新データを取得し、Excelの**シート「ALL」**を更新します。

### 修正点:
1. **Tickerの自動取得**: Yahoo Financeの検索APIを利用して、CUSIPまたは銘柄名からティッカーシンボルを自動的に推測・入力します。
2. **列の配置**: `CUSIP` の右隣に `Ticker` 列を自動作成し、空欄があれば補充します。

In [ ]:
# 必要なライブラリのインストール
!pip install requests pandas beautifulsoup4 openpyxl lxml -q

import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
from google.colab import drive
import time

# --- 設定 ---
CIK = "0001668527"
USER_AGENT = "Kei Sanada sdk7777@gmail.com"
SEC_HEADERS = {"User-Agent": USER_AGENT}

# Googleドライブをマウント
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

EXCEL_PATH = "/content/drive/MyDrive/History_Numerai_13F-HR.xlsx"
SHEET_NAME = "ALL"

### 1. ティッカー取得関数の実装（Yahoo Finance API を使用）

In [ ]:
def get_ticker_from_query(query):
    """
    Yahoo Financeの検索APIを使用してティッカーを取得します。
    """
    if not query: return ""
    url = f"https://query2.finance.yahoo.com/v1/finance/search?q={query}"
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        res = requests.get(url, headers=headers, timeout=10)
        data = res.json()
        if data.get('quotes'):
            # 米国市場（NYSE/NASDAQ等）のものを優先して探す
            for q in data['quotes']:
                symbol = q.get('symbol', '')
                if symbol and '.' not in symbol: # '.'がないものは通常米国主要株
                    return symbol
            return data['quotes'][0].get('symbol', "")
        return ""
    except:
        return ""

### 2. SECデータ取得

In [ ]:
def get_latest_data(cik):
    sub_url = f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json"
    r = requests.get(sub_url, headers=SEC_HEADERS)
    r.raise_for_status()
    recent = r.json()['filings']['recent']
    
    acc_num, report_date = None, None
    for i, form in enumerate(recent['form']):
        if form == '13F-HR':
            acc_num = recent['accessionNumber'][i].replace('-', '')
            report_date = recent['reportDate'][i]
            break
    if not acc_num: return None, None

    idx_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num}/index.json"
    idx_r = requests.get(idx_url, headers=SEC_HEADERS)
    idx_r.raise_for_status()
    
    xml_file = next((f['name'] for f in idx_r.json()['directory']['item'] if 'informationtable' in f['name'].lower() and f['name'].endswith('.xml')), None)
    if not xml_file:
        xml_file = next((f['name'] for f in idx_r.json()['directory']['item'] if f['name'].endswith('.xml') and 'primary' not in f['name'].lower()), None)

    xml_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num}/{xml_file}"
    print(f"最新データ取得中: {report_date}")
    xml_r = requests.get(xml_url, headers=SEC_HEADERS)
    xml_r.raise_for_status()
    
    soup = BeautifulSoup(xml_r.content, 'xml')
    results = []
    for info in soup.find_all(['infoTable', 'ns1:infoTable']):
        results.append({
            'CUSIP': info.find(['cusip', 'ns1:cusip']).text,
            'ISSUER NAME': info.find(['nameOfIssuer', 'ns1:nameOfIssuer']).text,
            'Value': int(info.find(['value', 'ns1:value']).text or 0)
        })
    
    df_new = pd.DataFrame(results).groupby(['CUSIP', 'ISSUER NAME'])['Value'].sum().reset_index()
    return df_new, report_date

df_latest, latest_date = get_latest_data(CIK)

### 3. Excel更新とティッカー検索の実行

In [ ]:
if df_latest is not None:
    if os.path.exists(EXCEL_PATH):
        xl = pd.ExcelFile(EXCEL_PATH)
        all_sheets = {name: xl.parse(name) for name in xl.sheet_names}
        
        if SHEET_NAME in all_sheets:
            df_all = all_sheets[SHEET_NAME]
            
            if 'Ticker' not in df_all.columns:
                df_all.insert(df_all.columns.get_loc('CUSIP') + 1, 'Ticker', "")
            
            df_all = pd.merge(df_all, df_latest.rename(columns={'Value': latest_date}), on=['CUSIP', 'ISSUER NAME'], how='outer')
            
            # Tickerが空の行（新規追加行や未取得行）を埋める
            print("未登録のTickerを検索中...")
            to_fill = df_all['Ticker'].isna() | (df_all['Ticker'].astype(str).str.strip() == "")
            count = 0
            for idx, row in df_all[to_fill].iterrows():
                # CUSIPで検索し、ダメなら銘柄名で検索
                ticker = get_ticker_from_query(row['CUSIP'])
                if not ticker:
                    ticker = get_ticker_from_query(row['ISSUER NAME'])
                
                if ticker:
                    df_all.at[idx, 'Ticker'] = ticker
                    count += 1
                time.sleep(0.5) # API負荷軽減
            
            print(f"{count} 件のTickerを新たに取得しました。")
            all_sheets[SHEET_NAME] = df_all
            
            with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
                for name, df in all_sheets.items():
                    df.to_excel(writer, sheet_name=name, index=False)
            print("Excelの保存が完了しました。")
        else: print(f"Error: {SHEET_NAME} sheet not found.")
    else: print("Excel file not found.")
else: print("Failed to fetch SEC data.")